<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-06/AI_Modul_6.4_Perbandingan_Performa_Model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 6.4: Perbandingan Performa Model - Uji Signifikansi Statistik & Evaluasi Multi-Kriteria
**Program Studi Agroteknologi, Agribisnis, dan Teknik Pertanian | INSTIPER Yogyakarta**

Modul ini memandu Anda dalam melakukan komparasi performa model kecerdasan buatan (*machine learning*) secara saintifik dan berstandar industri:
1. Menghitung dan memvisualisasikan Kurva ROC (*Receiver Operating Characteristic*) dan ROC-AUC lintas arsitektur model.
2. Menentukan titik ambang batas keputusan optimal (*Optimal Operating Threshold*) menggunakan statistik Youden's $J$.
3. Menguji signifikansi statistik performa antar-model menggunakan *Paired Student's t-Test* dengan koreksi dependensi Nadeau-Bengio.
4. Menerapkan uji non-parametrik *Wilcoxon Signed-Rank Test* dan *Uji Friedman* untuk multi-model.
5. Melakukan analisis kompromi multi-kriteria industri (*accuracy vs edge latency vs power trade-off*).

In [ ]:
import time
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, roc_auc_score, auc

print("Seluruh pustaka analisis performa model berhasil dimuat!")


## 1. Sintesis Dataset Agronomi: Deteksi Defisiensi Nutrisi Kanopi Kelapa Sawit
Kita membangkitkan $1.200$ sampel kanopi kebun sawit dengan $10$ fitur spektral multispektral UAV (seperti NDVI, GNDVI, NDRE, SAVI, MSAVI, EVI, dll.). Target biner:
- `0`: Kanopi Sehat / Nutrisi Optimal ($75\%$ populasi)
- `1`: Kanopi Mengalami Stres Nutrisi / Defisiensi Hara NPK ($25\%$ populasi)


In [ ]:
# Membangkitkan dataset sintetis realistis perkebunan
X, y = make_classification(
    n_samples=1200,
    n_features=10,
    n_informative=7,
    n_redundant=2,
    n_classes=2,
    weights=[0.75, 0.25],
    flip_y=0.03, # 3% label noise akibat variasi pencahayaan lapangan
    random_state=42
)

feature_names = ['NDVI', 'GNDVI', 'NDRE', 'SAVI', 'MSAVI', 'EVI', 'MCARI', 'TVI', 'BSI', 'REIP']
df_features = pd.DataFrame(X, columns=feature_names)
df_features['Status_Kanopi'] = y

print(f"Dimensi Dataset: {df_features.shape}")
print(f"Distribusi Kelas Status Kanopi:\n{df_features['Status_Kanopi'].value_counts(normalize=True).round(3)}")
print(df_features.head())


## 2. Pelatihan & Evaluasi 10-Fold Stratified Cross-Validation Multi-Model
Kita membandingkan 4 model dengan arsitektur berbeda:
1. **Regresi Logistik (Baseline Parametrik Linier)**
2. **Support Vector Machine (Kernel RBF)**
3. **Random Forest (Ansambel Bagging Pohon)**
4. **Gradient Boosting (Ansambel Boosting Pohon)**


In [ ]:
# Definisi 4 kandidat model
models = {
    'Logistic_Regression': LogisticRegression(max_iter=500, random_state=42),
    'SVM_RBF': SVC(probability=True, kernel='rbf', random_state=42),
    'Random_Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient_Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Penyimpanan hasil evaluasi
auc_results = {m: [] for m in models}
latency_results = {m: [] for m in models}

# List untuk interpolasi kurva ROC agregat
mean_fpr = np.linspace(0, 1, 100)
tprs_dict = {m: [] for m in models}

print("Memulai evaluasi 10-Fold Stratified Cross-Validation...")

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    for name, clf in models.items():
        # Latih model
        clf.fit(X_train, y_train)
        
        # Ukur latensi inferensi pada test set
        t_start = time.perf_counter()
        probs = clf.predict_proba(X_test)[:, 1]
        t_elapsed_ms = (time.perf_counter() - t_start) * 1000.0 / len(X_test)
        latency_results[name].append(t_elapsed_ms)
        
        # Hitung ROC-AUC
        score = roc_auc_score(y_test, probs)
        auc_results[name].append(score)
        
        # Simpan kurva ROC fold
        fpr, tpr, _ = roc_curve(y_test, probs)
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs_dict[name].append(interp_tpr)

df_auc = pd.DataFrame(auc_results)
df_latency = pd.DataFrame(latency_results)

print("\n=== RINGKASAN SKOR ROC-AUC (10-FOLD CV) ===")
summary_df = pd.DataFrame({
    'Rerata AUC': df_auc.mean(),
    'Std Dev AUC': df_auc.std(ddof=1),
    'Min AUC': df_auc.min(),
    'Max AUC': df_auc.max(),
    'Latensi Inferensi (ms/sampel)': df_latency.mean()
}).round(4)
print(summary_df)


## 3. Visualisasi Kurva ROC Agregat & Ambang Batas Optimal (Youden's J Index)
Kita memetakan kurva ROC komparatif rata-rata lintas 10 lipatan, lalu menghitung ambang batas operasional terbaik bagi model Gradient Boosting menggunakan indeks Youden:
$$J(\theta) = \text{TPR}(\theta) - \text{FPR}(\theta)$$


In [ ]:
plt.figure(figsize=(9, 7), dpi=150)
colors = {'Logistic_Regression': '#e65100', 'SVM_RBF': '#1565c0', 'Random_Forest': '#2e7d32', 'Gradient_Boosting': '#1b5e20'}
styles = {'Logistic_Regression': ':', 'SVM_RBF': '-.', 'Random_Forest': '--', 'Gradient_Boosting': '-'}

for name in models:
    mean_tpr = np.mean(tprs_dict[name], axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = auc(mean_fpr, mean_tpr)
    plt.plot(mean_fpr, mean_tpr, color=colors[name], linestyle=styles[name], lw=2.2,
             label=f'{name} (Rerata AUC = {mean_auc:.3f})')

# Diagonal tebakan acak
plt.plot([0, 1], [0, 1], color='#757575', lw=1.5, linestyle='--', label='Tebakan Acak (AUC = 0.500)')

# Hitung Youden's J pada model Gradient Boosting
gb_mean_tpr = np.mean(tprs_dict['Gradient_Boosting'], axis=0)
j_scores = gb_mean_tpr - mean_fpr
best_idx = np.argmax(j_scores)
opt_fpr = mean_fpr[best_idx]
opt_tpr = gb_mean_tpr[best_idx]

plt.scatter([opt_fpr], [opt_tpr], color='#c62828', s=100, zorder=5, edgecolor='black')
plt.annotate(f"Optimal Youden's J (Gradient Boosting)\nFPR={opt_fpr:.2f}, TPR={opt_tpr:.2f}",
             xy=(opt_fpr, opt_tpr), xytext=(opt_fpr + 0.15, opt_tpr - 0.12),
             arrowprops=dict(facecolor='#c62828', shrink=0.08, width=1.2, headwidth=6),
             fontsize=9, fontweight='bold', bbox=dict(boxstyle='round,pad=0.4', facecolor='#ffebee', edgecolor='#c62828'))

plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.05])
plt.xlabel('False Positive Rate (1 - Spesifisitas)', fontsize=11, fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivitas / Recall)', fontsize=11, fontweight='bold')
plt.title('Kurva ROC Komparasi Model Deteksi Stres Nutrisi Tanaman Sawit\n(Rerata 10-Fold CV)', fontsize=12, fontweight='bold')
plt.legend(loc='lower right', fontsize=9.5)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('docs/assets/roc_curve_output_nb64.png', dpi=150)
plt.close()
print("Plot kurva ROC berhasil dibuat dan disimpan.")


## 4. Uji Signifikansi Statistik: Paired t-Test Terkoreksi Nadeau-Bengio
Kita membandingkan performa antara **Gradient Boosting** dan **Random Forest**. Uji paired t-test standar mengasumsikan independensi lipatan, sedangkan validasi silang mengalami tumpang tindih data latih ($80-90\%$). Rumus varians terkoreksi Nadeau-Bengio:
$$\sigma_{\text{corr}}^2 = s_d^2 \left( \frac{1}{K} + \frac{n_{\text{test}}}{n_{\text{train}}} \right)$$


In [ ]:
def evaluate_paired_tests(scores_a, scores_b, n_samples, n_splits=10):
    scores_a = np.array(scores_a)
    scores_b = np.array(scores_b)
    d = scores_a - scores_b
    K = len(d)
    
    mean_d = np.mean(d)
    var_d = np.var(d, ddof=1)
    std_d = np.std(d, ddof=1)
    
    # 1. Standard Paired t-Test (Uncorrected)
    se_uncorrected = std_d / np.sqrt(K)
    t_std = mean_d / se_uncorrected
    p_std = 2 * (1 - stats.t.cdf(np.abs(t_std), df=K-1))
    
    # 2. Nadeau-Bengio Corrected t-Test
    n_test = n_samples / n_splits
    n_train = n_samples - n_test
    correction_factor = (1.0 / K) + (n_test / n_train)
    se_corrected = np.sqrt(var_d * correction_factor)
    t_nb = mean_d / se_corrected
    p_nb = 2 * (1 - stats.t.cdf(np.abs(t_nb), df=K-1))
    
    # 3. Wilcoxon Signed-Rank Test (Non-parametrik)
    w_stat, p_wilc = stats.wilcoxon(scores_a, scores_b)
    
    results = {
        'Rerata Selisih (A - B)': mean_d,
        'Std Dev Selisih': std_d,
        't-Stat Standar (Uncorrected)': t_std,
        'p-Value Standar': p_std,
        'Faktor Koreksi Nadeau-Bengio': correction_factor,
        't-Stat Nadeau-Bengio': t_nb,
        'p-Value Nadeau-Bengio': p_nb,
        'Wilcoxon W-Stat': w_stat,
        'Wilcoxon p-Value': p_wilc
    }
    return results

# Komparasi: Gradient_Boosting vs Random Forest
comp_gb_rf = evaluate_paired_tests(df_auc['Gradient_Boosting'], df_auc['Random_Forest'], n_samples=len(X))

print("=== HASIL UJI SIGNIFIKANSI STATISTIK (Gradient Boosting vs Random Forest) ===")
for k, v in comp_gb_rf.items():
    print(f"{k:32s}: {v:.5f}")

alpha = 0.05
print("\n=== KESIMPULAN INFERENSI STATISTIK ===")
if comp_gb_rf['p-Value Nadeau-Bengio'] < alpha:
    print(f"[SIGNIFIKAN] Model A (Gradient Boosting) terbukti lebih unggul secara statistik (p = {comp_gb_rf['p-Value Nadeau-Bengio']:.4f} < {alpha}).")
else:
    print(f"[TIDAK SIGNIFIKAN] Keunggulan Gradient Boosting terhadap Random Forest belum dapat dipastikan secara statistik pada alpha = {alpha} (p = {comp_gb_rf['p-Value Nadeau-Bengio']:.4f} >= {alpha}).")


## 5. Uji Multi-Model: Uji Friedman & Ranking Algoritma
Jika kita membandingkan lebih dari 2 model sekaligus, kita menggunakan **Uji Friedman** berbasis peringkat non-parametrik untuk menghindari inflasi Galat Tipe I.


In [ ]:
# Uji Friedman lintas seluruh 4 model
stat_f, p_f = stats.friedmanchisquare(
    df_auc['Logistic_Regression'],
    df_auc['SVM_RBF'],
    df_auc['Random_Forest'],
    df_auc['Gradient_Boosting']
)

print(f"Statistik Chi-Square Friedman: {stat_f:.4f}")
print(f"p-Value Uji Friedman        : {p_f:.4e}")

# Peringkat rata-rata per lipatan
ranks = df_auc.rank(axis=1, ascending=False)
mean_ranks = ranks.mean()
print("\n=== PERINGKAT RATA-RATA MODEL (1 = Terbaik, 4 = Terburuk) ===")
print(mean_ranks.sort_values())


## 6. Analisis Kompromi Multi-Kriteria Industri (The Accuracy-Efficiency Trade-off)
Di lapangan, model akan dipasang pada mikrokomputer *edge computing* di atas drone penyemprotan kebun sawit. Kita menghitung **Skor Utilitas Komposit**:
$$U(m) = w_1 \cdot \text{AUC}(m) + w_2 \cdot \left(1 - \frac{\text{Latensi}(m)}{\text{Latensi}_{\max}}\right) + w_3 \cdot \left(1 - \frac{\text{Memori}(m)}{\text{Memori}_{\max}}\right)$$


In [ ]:
# Parameter bobot manajerial agribisnis
w_auc = 0.50
w_lat = 0.30
w_mem = 0.20

lat_max = 5.0  # ms/sampel batas toleransi kontrol drone
mem_max = 50.0 # MB batas RAM alokasi model edge

# Asumsi estimasi konsumsi memori model
mem_estimates = {
    'Logistic_Regression': 0.5,
    'SVM_RBF': 15.0,
    'Random_Forest': 38.0,
    'Gradient_Boosting': 6.5
}

utility_scores = {}
for m in models:
    score_auc = df_auc[m].mean()
    lat = df_latency[m].mean()
    mem = mem_estimates[m]
    
    u_lat = max(0.0, 1.0 - (lat / lat_max))
    u_mem = max(0.0, 1.0 - (mem / mem_max))
    
    utility = (w_auc * score_auc) + (w_lat * u_lat) + (w_mem * u_mem)
    utility_scores[m] = {
        'ROC_AUC': round(score_auc, 4),
        'Latensi_ms': round(lat, 3),
        'Memori_MB': mem,
        'Skor_Utilitas_U': round(utility, 4)
    }

df_utility = pd.DataFrame(utility_scores).T
print("=== MATRIKS KEPUTUSAN MULTI-KRITERIA SELEKSI MODEL PERTANIAN ===")
print(df_utility.sort_values(by='Skor_Utilitas_U', ascending=False))

# Plot visualisasi skor utilitas
plt.figure(figsize=(8, 4.5), dpi=150)
bars = plt.bar(df_utility.index, df_utility['Skor_Utilitas_U'], color=['#388e3c', '#1976d2', '#f57c00', '#7b1fa2'], alpha=0.85, edgecolor='black')
plt.ylabel('Skor Utilitas Terbobot U(m)', fontsize=11, fontweight='bold')
plt.title('Evaluasi Multi-Kriteria untuk Deployment Edge AI Drone Kebun Sawit', fontsize=12, fontweight='bold')
plt.ylim([0, 1.0])
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f'{yval:.3f}', ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('docs/assets/utility_score_output_nb64.png', dpi=150)
plt.close()
print("Plot skor utilitas berhasil dibuat dan disimpan!")
